In [3]:
import numpy as np
from scipy.integrate import quad
from deap import algorithms, base, creator, tools
import matplotlib.pyplot as plt

In [5]:
data = np.loadtxt('synthetic_magnetostratigraphy.txt')  # Or your saved file
z_obs, e_obs, polarity_obs = data[:, 0], data[:, 1], data[:, 2]

In [6]:
c1, c2 = 59.8, 60.1

In [11]:
def get_lock_depth_from_params(params):
    a1,a2,b1,b2 = params
    lock_in_depth = [(b1-2)/a1,(b1+2)/a1,(b2-2)/a2,(b2+2)/a2]
    return lock_in_depth

In [12]:
def get_params_from_depths(lock_in_depth):
    l0,l1,l2,l3 = lock_in_depth
    params = [4/(l1-l0),4/(l3-l2),2*(l1+l0)/(l1-l0),2*(l3+l2)/(l3-l2)]
    return params

In [8]:
def e(z):
    idx = np.where(np.isclose(z_obs, z))[0][0]
    return 1 - e_obs[idx]  # Assuming e_obs is (1 - fraction_data)

def l_diff(z, s, a1, b1, a2, b2):
    term1 = (a1 * np.exp(-a1 * s + b1)) / (1 + np.exp(-a1 * s + b1))**2
    term2 = (a2 * np.exp(-a2 * s + b2)) / (1 + np.exp(-a2 * s + b2))**2
    return e(z) * term1 + (1 - e(z)) * term2

def H(z,c1,c2):
    # Ensure z is treated properly, even if it's an array
    return -np.tanh(((c2 + c1) * (z - (c2 + c1) / 2)) / (c2 - c1))

def get_magnetisation(z, params, c1, c2):
    a1, a2, b1, b2 = params
    M = np.array([quad(lambda s: H(z_i - s, c1, c2) * l_diff(z_i, s, a1, b1, a2, b2), 0, z_i)[0] 
                  for z_i in z])
    return np.tanh(M * 1e3)

def get_polarity(z, params, c1, c2, threshold=0.5):
    M = get_magnetisation(z, params, c1, c2)
    return np.where(M > threshold, 1, np.where(M < -threshold, -1, 0))

In [9]:
def fitness(params, z_obs, polarity_obs, c1, c2):
    modeled_polarity = get_polarity(z_obs, params, c1, c2)
    return np.sum(modeled_polarity != polarity_obs),  # DEAP requires a tuple

# DEAP framework setup
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_float", np.random.uniform, low=0.1, high=10)  # Bounds for a1, a2, b1, b2
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=4)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", fitness, z_obs=z_obs, polarity_obs=polarity_obs, c1=c1, c2=c2)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

In [ ]:
population = toolbox.population(n=50)
result = algorithms.eaSimple(population, toolbox, cxpb=0.5, mutpb=0.2, ngen=100, verbose=True)

# Best solution
best_individual = tools.selBest(population, k=1)[0]
best_params = best_individual
best_lock_in_depths = get_lock_depth_from_params(best_params)  # Your existing function
print("Recovered lock-in depths:", best_lock_in_depths)